# Erdos-Straus Modular Sieve -- SageMaker Studio Lab

**Guinea Pig Trench LLC**

---

Modular sieve verification. SageMaker Studio Lab: 4 vCPUs, 16GB RAM, 12-hour sessions, persistent storage.

GitHub: [Commencethescourge/erdos-straus-solver](https://github.com/Commencethescourge/erdos-straus-solver)

In [ ]:
# === Setup & Data Download ===
import os, subprocess, multiprocessing as mp

cpu_count = mp.cpu_count()
print(f'CPU cores: {cpu_count}')

SIEVE_DIR = os.path.expanduser('~/sieve_data')
RESULTS_DIR = os.path.expanduser('~/erdos_straus')
os.makedirs(SIEVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

for fname in ['Residues.txt', 'Filters.txt']:
    dest = os.path.join(SIEVE_DIR, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f'{fname} already downloaded ({os.path.getsize(dest)/1e6:.1f}MB)')
        continue
    url = f'https://github.com/esc-paper/erdos-straus/raw/main/section1/resources/{fname}'
    print(f'Downloading {fname}...')
    subprocess.run(['wget', '-q', '-O', dest, url], check=True)
    print(f'  Done: {os.path.getsize(dest)/1e6:.1f}MB')

print(f'\nSieve data: {SIEVE_DIR}')
print(f'Results: {RESULTS_DIR} (persistent)')

In [ ]:
# === Sieve Solver (self-contained) ===
import csv, math, time
import multiprocessing as mp

G_8 = 25_878_772_920
RES_PATH = os.path.join(SIEVE_DIR, 'Residues.txt')
FILT_PATH = os.path.join(SIEVE_DIR, 'Filters.txt')

def load_residues(path=None):
    path = path or RES_PATH
    with open(path) as f:
        return list(map(int, f.read().split()))

def load_filters(path=None):
    path = path or FILT_PATH
    filters = []
    current_prime = None
    current_residues = []
    with open(path) as f:
        for line in f:
            for token in line.split():
                n = int(token)
                if n == -1:
                    if current_prime is not None:
                        filters.append((current_prime, frozenset(current_residues)))
                    current_prime = None
                    current_residues = []
                elif current_prime is None:
                    current_prime = n
                else:
                    current_residues.append(n)
    if current_prime is not None:
        filters.append((current_prime, frozenset(current_residues)))
    return filters

_w_residues = None
_w_filters = None

def _init_worker(res_path, filt_path):
    global _w_residues, _w_filters
    _w_residues = load_residues(res_path)
    _w_filters = load_filters(filt_path)

def _check_batch(k):
    survivors = []
    for r in _w_residues:
        n = r + k * G_8
        for p, excluded in _w_filters:
            if n % p in excluded:
                break
        else:
            survivors.append(n)
    return survivors

def check_batch(k):
    return (k, _check_batch(k))

def is_prime(n):
    if n < 2: return False
    if n < 4: return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0: return False
        i += 6
    return True

res = load_residues()
filt = load_filters()
print(f'Solver loaded: {len(res):,} residues, {len(filt):,} filters')
del res, filt

In [ ]:
# === Configure ===
K_START = 1934     # <-- edit
K_END   = 2900     # <-- edit (SageMaker's share of 10^14)
WORKERS = max(2, cpu_count)

total_batches = K_END - K_START + 1
n_max = (K_END + 1) * G_8
print(f'Batches: {K_START:,} to {K_END:,} ({total_batches:,} total)')
print(f'Verification up to: ~{n_max:.2e}')
print(f'Workers: {WORKERS}')

In [ ]:
# === Stats & Download ===
if os.path.exists(checkpoint):
    batches = set()
    survivors = 0
    with open(checkpoint) as f:
        for row in csv.DictReader(f):
            batches.add(int(row['batch_k']))
            if row.get('is_prime', '').lower() == 'true':
                survivors += 1
    k_max = max(batches) if batches else 0
    print(f'Batches completed: {len(batches):,}')
    print(f'Verified up to: ~{(k_max+1)*G_8:.2e}')
    print(f'Prime survivors: {survivors}')
    if survivors == 0:
        print('Conjecture holds for this range!')
    print(f'\nResults at: {checkpoint}')
    print(f'  (persistent across sessions)')
    print(f'\nTo centralize: download CSV and drop into Google Drive > erdos_straus/')

In [ ]:
# === Stats ===
if os.path.exists(checkpoint):
    batches = set()
    survivors = 0
    with open(checkpoint) as f:
        for row in csv.DictReader(f):
            batches.add(int(row['batch_k']))
            if row.get('is_prime', '').lower() == 'true':
                survivors += 1
    k_max = max(batches) if batches else 0
    print(f'Batches completed: {len(batches):,}')
    print(f'Verified up to: ~{(k_max+1)*G_8:.2e}')
    print(f'Prime survivors: {survivors}')
    if survivors == 0:
        print('Conjecture holds for this range!')